# ELMUR-KLPO Step 3 on Colab

Run this notebook with a GPU runtime. The RememberColor demonstrations are downloaded from the pinned Hugging Face revision; the archive and checkpoints stay under `/content` and are not committed to GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT = Path('/content/ELMUR-KLPO')
OUTPUT = Path('/content/drive/MyDrive/elmur-klpo-runs/step3_upstream_rc3')
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
print(PROJECT, OUTPUT)

In [ ]:
!rm -rf /content/ELMUR-KLPO
!git clone https://github.com/getFaster/elmur-klpo-colab.git /content/ELMUR-KLPO
%cd /content/ELMUR-KLPO
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
os.environ['OUTPUT'] = '/content/drive/MyDrive/elmur-klpo-runs/step3_upstream_rc3'
!uv sync

In [ ]:
!pip -q install huggingface_hub
import hashlib
from pathlib import Path
from huggingface_hub import hf_hub_download

download_dir = PROJECT / 'runs/step3_data/remember_color_download'
download_dir.mkdir(parents=True, exist_ok=True)
archive = Path(hf_hub_download(
    repo_id='avanturist/mikasa-robo',
    repo_type='dataset',
    filename='RememberColor3-v0.zip',
    revision='c362f52143c61e7f778bdb5d410ce4c22a7185d4',
    local_dir=download_dir,
))
expected = '388b16d0984cb19d6a6cd20fe0029a67fef100aed378a195435f1f0acf5b7d82'
digest = hashlib.sha256()
with archive.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        digest.update(chunk)
assert digest.hexdigest() == expected, digest.hexdigest()
print(f'verified {archive} ({archive.stat().st_size / 1e9:.2f} GB)')

In [ ]:
import json
import shutil

data_root = PROJECT / 'runs/step3_data/remember_color_original'
data_root.mkdir(parents=True, exist_ok=True)
if not (data_root / 'RememberColor3-v0').exists():
    shutil.unpack_archive(str(archive), str(data_root))
config_path = PROJECT / 'configs/step3_remembercolor_reader.json'
config = json.loads(config_path.read_text())
entry = config['tasks'][0]
data_dir = str((data_root / 'RememberColor3-v0').resolve())
entry['preparation']['demonstrations']['directory'] = data_dir
entry['preparation']['demonstrations']['manifest'] = str((PROJECT / 'experiments/step3/remember_color_dataset.json').resolve())
manifest_path = PROJECT / 'experiments/step3/remember_color_dataset.json'
manifest = json.loads(manifest_path.read_text())
manifest['data_directory'] = data_dir
manifest['archive_path'] = str(archive.resolve())
config_path.write_text(json.dumps(config, indent=2) + '\n')
manifest_path.write_text(json.dumps(manifest, indent=2) + '\n')
print(f'{len(list((data_root / "RememberColor3-v0").glob("train_data_*.npz")))} episodes ready')

In [ ]:
!nvidia-smi
!uv run python -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"
!uv run pytest tests/test_offline_preparation.py tests/test_step3_upstream.py -q

In [ ]:
!uv run python -m elmur_klpo.experiment --config configs/step3_remembercolor_reader.json --output "$OUTPUT" --device cuda

The full reader budget is 9,600 optimizer updates and can take substantial GPU time. If Colab disconnects, rerun the final cell with `--resume`; the checkpoint and history are on Google Drive.

In [ ]:
!uv run python -m elmur_klpo.experiment --config configs/step3_remembercolor_reader.json --output "$OUTPUT" --device cuda --resume